In [2]:
import os, json, time, numpy as np, pandas as pd, dash
import plotly.express as px, plotly.graph_objs as go, dash_bootstrap_components as dbc
from dash import Input, Output, State, dcc, html, dash_table, ALL, MATCH, ctx
#region Start
app = dash.Dash(
    __name__,
    external_stylesheets=[
        dbc.themes.BOOTSTRAP, #Wygląd
        "https://cdn.jsdelivr.net/npm/bootstrap-icons@1.11.0/font/bootstrap-icons.css"
    ],
    suppress_callback_exceptions=True
)
DATA_FOLDER = 'Dane Dash/' #Folder do zczytania gotowych danych
frames = []  
for filename in os.listdir(DATA_FOLDER):
    if filename.endswith('.json'):
        filepath = os.path.join(DATA_FOLDER, filename)
        frames.append(pd.read_json(filepath))
df = pd.concat(frames, ignore_index=True)

In [14]:
df = df[df.osoba.isin(['A','AK','J'])]

In [15]:
# Zmiana na czas polski
timestamp = df['timestamp']
timestamp_pl = timestamp.dt.tz_convert('Europe/Warsaw')
df['timestamp_pl'] = timestamp_pl
df['date'] = timestamp_pl.dt.date
df.sample(2)


,timestamp,duration,app,title,status,category,osoba,device,timestamp_pl,date
46820,2025-12-30 15:11:33.936000+00:00,1.051,wincompose.exe,NaN,not-afk,inne,J,PC,2025-12-30 16:11:33.936000+01:00,2025-12-30
28319,2026-01-01 00:35:03.920000+00:00,30.930,brave.exe,(35) THE WORST SION I'VE EVER SEEN! (not me bt...,not-afk,oglądanie filmów,A,PC,2026-01-01 01:35:03.920000+01:00,2026-01-01


In [16]:
Suma_z_dnia = df.groupby(['osoba','date'])['duration'].sum().reset_index().sort_values('duration',ascending=False)
Suma_z_dnia['duration'] = round(Suma_z_dnia['duration'] / 3600,2)
Suma_z_dnia = Suma_z_dnia.sort_values(['osoba','date'])
Suma_z_dnia.head()

,osoba,date,duration
0,A,2025-12-13,5.76
1,A,2025-12-14,8.28
2,A,2025-12-15,14.40
3,A,2025-12-16,5.25
4,A,2025-12-17,5.47


In [17]:
# Wszystkie dni z zakresu
osoby = Suma_z_dnia['osoba'].unique()
daty = pd.date_range(Suma_z_dnia['date'].min(), Suma_z_dnia['date'].max())

full_index = pd.MultiIndex.from_product([osoby, daty], names=['osoba', 'date'])
Suma_z_dnia = Suma_z_dnia.set_index(['osoba', 'date']).reindex(full_index, fill_value=0).reset_index()
Suma_z_dnia.sort_values(["osoba","date"]).reset_index(drop=True)

,osoba,date,duration
0,A,2025-12-13,5.76
1,A,2025-12-14,8.28
2,A,2025-12-15,14.40
3,A,2025-12-16,5.25
4,A,2025-12-17,5.47
...,...,...,...
82,J,2026-01-06,1.23
83,J,2026-01-07,0.00
84,J,2026-01-08,0.00
85,J,2026-01-09,0.00


In [23]:
# zakładam, że df['date'] jest typu datetime
fig = px.line(
    Suma_z_dnia,
    x='date',
    y='duration',
    color='osoba',           # każda osoba będzie miała osobną linię
    labels={'date':'Data', 'duration':'Czas aktywności (godz.)', 'osoba':'Osoba'},
    title='Średnia aktywność dzienna',
           template='plotly_white',

)

# wymuszenie chronologicznej kolejności X, jeśli daty są datetime to Plotly zrobi to automatycznie
fig.update_xaxes(dtick="D1", tickformat="%Y-%m-%d")  # co dzień
fig.update_layout(xaxis_tickangle=-45)  # dla czytelności etykiet dat
fig.update_traces(line=dict(width=4))  # np. 4 piksele

fig.show()


In [24]:
# zakładam, że df['date'] jest typu datetime
fig = px.bar(
    Suma_z_dnia,
    x='date',
    y='duration',
    color='osoba',           # każda osoba będzie miała osobną linię
    labels={'date':'Data', 'duration':'Czas aktywności (godz.)', 'osoba':'Osoba'},
    title='Średnia aktywność dzienna',
           template='plotly_white',

)

# wymuszenie chronologicznej kolejności X, jeśli daty są datetime to Plotly zrobi to automatycznie
fig.update_xaxes(dtick="D1", tickformat="%Y-%m-%d")  # co dzień
fig.update_layout(xaxis_tickangle=-45,  barmode='group')  # dla czytelności etykiet dat

fig.show()


In [25]:
osoba = 'J'
Suma_osoby = Suma_z_dnia[Suma_z_dnia.osoba == osoba]

fig = px.bar(
    Suma_osoby,
    x='date',
    y='duration',
    color='osoba',           # każda osoba będzie miała osobną linię
    labels={'date':'Data', 'duration':'Czas aktywności (godz.)', 'osoba':'Osoba'},
    title='Średnia aktywność dzienna',
           template='plotly_white',

)

# wymuszenie chronologicznej kolejności X, jeśli daty są datetime to Plotly zrobi to automatycznie
fig.update_xaxes(dtick="D1", tickformat="%Y-%m-%d")  # co dzień
fig.update_layout(xaxis_tickangle=-45,  barmode='group')  # dla czytelności etykiet dat

fig.show()
